In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats


In [2]:
PROJ_ROOT = Path("data/processed")  # or Path(r"d:\Git_repository\result folder\data\processed")
DS_ROOT   = PROJ_ROOT / "downsampled"

GROUPS = ["conventional", "vr_high", "vr_low"]
STRESS_FOLDER = "stress"
SIGNALS = ["SCL", "HR", "PVA", "TEMP", "MOTION"]
MAX_SECONDS = 240  # analyze first 240 s of each phase

COLLAPSE_MAP = {
    "conventional": "Conventional",
    "vr_high": "VR",
    "vr_low": "VR",
}


In [3]:
def load_phase_mean(participant_id: str, phase_folder: str) -> dict:
    """
    Load one participant's CSV for a given phase and return
    mean(0-240s) per signal.

    Returns dict: { "SCL_mean": ..., "HR_mean": ..., ... }
    Missing files -> None
    """
    fpath = DS_ROOT / phase_folder / f"{participant_id}.csv"
    if not fpath.exists():
        return None

    df = pd.read_csv(fpath)
    if "sec" not in df.columns:
        return None

    df = df[df["sec"] < MAX_SECONDS].copy()
    out = {}
    for sig in SIGNALS:
        if sig in df.columns:
            out[f"{sig}_mean"] = df[sig].mean()
        else:
            out[f"{sig}_mean"] = np.nan
    return out


def build_delta_table() -> pd.DataFrame:
    """
    Loop over all meditation groups, find participants, load stress+meditation
    means, and compute Δ = med - stress for each signal.

    Returns a DataFrame with columns:
        participant, group, group_collapsed,
        <SIG>_stress_mean, <SIG>_med_mean, <SIG>_delta
    """
    rows = []

    for g in GROUPS:
        gdir = DS_ROOT / g
        if not gdir.exists():
            print(f"⚠️ Skipping {g}: folder not found")
            continue

        for f in sorted(gdir.glob("*.csv")):
            pid = f.stem  # participant ID from filename, e.g., "109"

            # Load med phase
            med_means = load_phase_mean(pid, g)
            if med_means is None:
                print(f"⚠️ No med file for participant {pid} in group {g}")
                continue

            # Load stress phase
            stress_means = load_phase_mean(pid, STRESS_FOLDER)
            if stress_means is None:
                print(f"⚠️ No stress file for participant {pid}, skipping")
                continue

            row = {
                "participant": pid,
                "group": g,
                "group_collapsed": COLLAPSE_MAP.get(g, None),
            }

            # For each signal, store stress_mean, med_mean, delta
            for sig in SIGNALS:
                s_key = f"{sig}_stress_mean"
                m_key = f"{sig}_med_mean"
                d_key = f"{sig}_delta"

                s_val = stress_means.get(f"{sig}_mean", np.nan)
                m_val = med_means.get(f"{sig}_mean", np.nan)
                row[s_key] = s_val
                row[m_key] = m_val
                if np.isfinite(s_val) and np.isfinite(m_val):
                    row[d_key] = m_val - s_val
                else:
                    row[d_key] = np.nan

            rows.append(row)

    df = pd.DataFrame(rows)
    # Drop participants without collapsed group mapping
    df = df.dropna(subset=["group_collapsed"])
    return df


def cohen_d_independent(x, y):
    """Cohen's d for independent samples."""
    x = np.asarray(x)
    y = np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan
    pooled_sd = np.sqrt(((nx - 1)*x.var(ddof=1) + (ny - 1)*y.var(ddof=1)) / (nx + ny - 2))
    if pooled_sd == 0:
        return np.nan
    return (x.mean() - y.mean()) / pooled_sd


def cohen_d_paired(x, y):
    """Cohen's d for paired samples (mean diff / sd of diff)."""
    x = np.asarray(x)
    y = np.asarray(y)
    if len(x) < 2:
        return np.nan
    diff = y - x
    return diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) != 0 else np.nan


def rank_biserial_from_mwu(U, n1, n2):
    """Rank-biserial correlation from Mann–Whitney U."""
    return 1 - 2 * U / (n1 * n2)


# -------------------------------------------------------------------
# STATS
# -------------------------------------------------------------------
def run_within_group_tests(df: pd.DataFrame):
    """
    For each group (conventional, vr_high, vr_low) and each signal,
    compare stress vs meditation using paired t-test or Wilcoxon.
    """
    print("\n" + "="*60)
    print("WITHIN-GROUP: STRESS vs MEDITATION (per group)")
    print("="*60)

    for g in GROUPS:
        sub = df[df["group"] == g]
        if sub.empty:
            continue

        print(f"\n--- Group: {g} ---")
        for sig in SIGNALS:
            s_col = f"{sig}_stress_mean"
            m_col = f"{sig}_med_mean"

            tmp = sub[[s_col, m_col]].dropna()
            if tmp.empty or len(tmp) < 3:
                print(f"  {sig}: not enough data ({len(tmp)} participants)")
                continue

            x = tmp[s_col].values
            y = tmp[m_col].values
            diff = y - x

            # Normality on diff
            sh = stats.shapiro(diff)
            is_normal = sh.pvalue > 0.05

            if is_normal:
                tstat, pval = stats.ttest_rel(x, y)
                d = cohen_d_paired(x, y)
                test_name = "paired t-test"
            else:
                tstat, pval = stats.wilcoxon(x, y, zero_method="wilcox", alternative="two-sided")
                d = np.nan  # we can still report median diff if you want
                test_name = "Wilcoxon signed-rank"

            print(
                f"  {sig}: {test_name} | n={len(tmp)}, "
                f"mean_stress={x.mean():.3f}, mean_med={y.mean():.3f}, "
                f"p={pval:.4f}, d={d:.3f}"
            )


def run_between_group_delta_tests(df: pd.DataFrame):
    """
    Collapse groups to Conventional vs VR and compare Δ = med - stress.
    """
    print("\n" + "="*60)
    print("BETWEEN-GROUP: Δ (meditation - stress) Conventional vs VR")
    print("="*60)

    for sig in SIGNALS:
        d_col = f"{sig}_delta"
        tmp = df[["group_collapsed", d_col]].dropna()
        if tmp.empty:
            print(f"\n{sig}: no data.")
            continue

        gA = tmp[tmp["group_collapsed"] == "Conventional"][d_col].values
        gB = tmp[tmp["group_collapsed"] == "VR"][d_col].values

        if len(gA) < 3 or len(gB) < 3:
            print(f"\n{sig}: not enough data (Conventional={len(gA)}, VR={len(gB)})")
            continue

        # Normality checks
        shA = stats.shapiro(gA)
        shB = stats.shapiro(gB)
        normalA = shA.pvalue > 0.05
        normalB = shB.pvalue > 0.05

        if normalA and normalB:
            tstat, pval = stats.ttest_ind(gA, gB, equal_var=False)
            d = cohen_d_independent(gA, gB)
            test_name = "independent t-test"
            extra = f", d={d:.3f}"
        else:
            # Mann–Whitney U
            U, pval = stats.mannwhitneyu(gA, gB, alternative="two-sided")
            r = rank_biserial_from_mwu(U, len(gA), len(gB))
            test_name = "Mann–Whitney U"
            extra = f", r_rb={r:.3f}"

        print(
            f"\n{sig}: {test_name} | "
            f"n_Conv={len(gA)}, n_VR={len(gB)}, "
            f"meanΔ_Conv={gA.mean():.3f}, meanΔ_VR={gB.mean():.3f}, "
            f"p={pval:.4f}{extra}"
        )



In [4]:
def main():
    df = build_delta_table()
    if df.empty:
        print("❌ No data found. Check DS_ROOT and folder structure.")
        return

    print("✅ Built delta table with shape:", df.shape)
    print(df.head())

    # Save table for reference
    out_path = PROJ_ROOT / "summaries" / "delta_table.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print("💾 Saved delta_table.csv to:", out_path)

    # Run tests
    run_within_group_tests(df)
    run_between_group_delta_tests(df)


if __name__ == "__main__":
    main()

✅ Built delta table with shape: (47, 18)
  participant         group group_collapsed  SCL_stress_mean  SCL_med_mean  \
0         100  conventional    Conventional         4.082981      6.426633   
1         103  conventional    Conventional         0.353764      0.383518   
2         105  conventional    Conventional         1.834040      2.268427   
3         107  conventional    Conventional         0.452854      1.449298   
4         108  conventional    Conventional         0.766397      0.842263   

   SCL_delta  HR_stress_mean  HR_med_mean   HR_delta  PVA_stress_mean  \
0   2.343652       87.495267    82.101899  -5.393368        23.611299   
1   0.029754       95.049733    78.744955 -16.304779        19.625119   
2   0.434387      102.569581    81.594966 -20.974615        23.174870   
3   0.996444       67.951966    68.263721   0.311755         7.435297   
4   0.075866       67.972253    70.989692   3.017438        31.992732   

   PVA_med_mean  PVA_delta  TEMP_stress_mean  TEMP_